### imports

In [2]:
import json
import pickle
from datetime import datetime as dt

import numpy as np
import pandas as pd

### functions

In [8]:
# use POSIX timestamp of first experiment page as start time --
# `beginhit` field records when prep screen was loaded, which was
# often done in advance
exp_start_time = lambda x: x['data'][0]['dateTime']

# have to hack the datetime conversion from the google forms -- 
# Google forms convert all dates to current time zone upon 
# downloading, so participants collected during EDT are shown 
# as EST equivalent. pandas default date_parser doesn't handle 
# this weird behavior correctly
parse_posix_ms = lambda x: pd.to_datetime(x).tz_localize(None).tz_localize('EST').timestamp()*1000

### load data

In [37]:
# load in psiturk data
rm1df = pd.read_json('../../../data/db/exported/room1-3.8.19.json')
rm2df = pd.read_json('../../../data/db/exported/room2-3.5.19.json')

# drop runs that didn't finish
rm1df = rm1df[rm1df.status != 1].reset_index(drop=True)
rm2df = rm2df[rm2df.status != 1].reset_index(drop=True)

# remove test runs for each room
rm1df = rm1df.loc[1:]
rm2df = rm2df.loc[1:]

# keep relevant columns
rm1df = rm1df[['uniqueid','datastring']]
rm2df = rm2df[['uniqueid','datastring']]

# load json string
rm1df['datastring'] = rm1df['datastring'].apply(json.loads)
rm2df['datastring'] = rm2df['datastring'].apply(json.loads)

# record start time
rm1df['beginhit'] = rm1df['datastring'].apply(exp_start_time)
rm2df['beginhit'] = rm2df['datastring'].apply(exp_start_time)

# add test room column
rm1df['testroom'] = 1
rm2df['testroom'] = 2


# concatenate testroom dataframes, order by start time
expdf = pd.concat([rm1df, rm2df], ignore_index=True).sort_values('beginhit').reset_index(drop=True)

# # drop subjects 
# expdf = expdf.loc[expdf.index != 7].reset_index(drop=True) # experiment error
# expdf = expdf.loc[expdf.index != 56].reset_index(drop=True) # no-show for part 2
# expdf = expdf.loc[expdf.index != 62].reset_index(drop=True) # no-show for part 2

In [38]:
expdf.head()

,uniqueid,datastring,beginhit,testroom
0,debugIEH2T:debugDLVLJ,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539368162836,1
1,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539371956776,1
2,debugYQfMB:debugxg7il,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539372566510,2
3,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539375821845,1
4,debug92cgv:debugvdAIT,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539376317256,2


In [39]:
# load pre/post questionnaire responses
preqdf = pd.read_csv('../../../data/google-form-data/Pre-experiment Questionnaire.csv')
preqdf = preqdf.rename(index=str, columns={'Timestamp':'preqtime'})
preqdf['preqtime'] = preqdf['preqtime'].apply(parse_posix_ms)


postqdf = pd.read_csv('../../../data/google-form-data/Post-experiment questionnaire.csv')
postqdf = postqdf.rename(index=str, columns={'Timestamp':'postqtime'})
postqdf['postqtime'] = postqdf['postqtime'].apply(parse_posix_ms)

# exclude test runs
preqdf = preqdf.dropna(subset=['Subject ID']).reset_index(drop=True)
postqdf = postqdf.dropna(subset=['Subject ID']).reset_index(drop=True)

In [50]:
# # convert from datetime to POSIX time **SERIES.APPLY(DT.DT.TIMESTAMP).MULTIPLY(1000) DOES NOT WORK**
# # add 3 hours to convert from UTC to ET
# # subtract 1 hours to EST times to account for datetime handling of DST
# newpretimestamp = pd.Series([0]*len(preqdf['preqtime']))
# for ix, val in enumerate(newpretimestamp):
#     newpretimestamp[ix] = preqdf['preqtime'][ix].timestamp()*1000
#     if ix <= 92:
#         newpretimestamp[ix] += 3.6e6

# preqdf['preqtime'] = newpretimestamp

# newposttimestamp = pd.Series([0]*len(postqdf['postqtime']))
# for ix, val in enumerate(newposttimestamp):
#     newposttimestamp[ix] = postqdf['postqtime'][ix].timestamp()*1000
#     if ix <= 92:
#         newposttimestamp[ix] += 3.6e6

# postqdf['postqtime'] = newposttimestamp

In [51]:
preqdf.shape

(136, 18)

In [52]:
expdf.shape

(132, 4)

In [23]:
# remove dropped subjects from google form
# dropids = ['MD-102218-B-04','MD-020119-A-01','MD-102318-A-01','MD-101318-A-05','MD-1011318-A-05','MD-020119-B-01',
#            'MD-101218-B-04']

dropids = ['MD-102218-B-04','MD-020119-A-01','MD-102318-A-01','MD-101318-A-05','MD-1011318-A-05','MD-020119-B-01',
           'MD-101218-B-04', 'MD-013119-A-01', 'MD-102218-A-04', ]

for dropid in dropids:
    preqdf = preqdf[preqdf['Subject ID'] != dropid]
    postqdf = postqdf[postqdf['Subject ID'] != dropid]
        
preqdf.reset_index(drop=True, inplace=True)
postqdf.reset_index(drop=True, inplace=True)

In [24]:
# fix mistaken day/repeat ID assignments

preqdf.at[76,'Subject ID'] = 'MD-102018-A-03'
postqdf.at[76,'Subject ID'] = 'MD-102018-A-03'

preqdf.at[77,'Subject ID'] = 'MD-102218-A-06'
postqdf.at[77,'Subject ID'] = 'MD-102218-A-06'

preqdf.at[85,'Subject ID'] = 'MD-102218-B-06'
postqdf.at[85,'Subject ID'] = 'MD-102218-B-06'

preqdf.at[89,'Subject ID'] = 'MD-013119-A-02'
postqdf.at[90,'Subject ID'] = 'MD-013119-A-02'

preqdf.at[123,'Subject ID'] = 'MD-022819-B-01'
postqdf.at[123,'Subject ID'] = 'MD-022819-B-01'

### for mapping between Google Forms with experiment IDs and SQLite databases with PsiTurk IDs

In [25]:
# add empty columns from pre/postquestionnaires to expdf
newcols = pd.unique(np.concatenate([i.columns.values for i in [preqdf,postqdf]]))
expdf = pd.concat([expdf, pd.DataFrame(columns=newcols)], sort=False)

# separate session 1 and 2 postquestionnaires
ses1postqdf = postqdf.drop_duplicates('Subject ID')
tempdf = postqdf.copy(deep=True)
rowsin1 = [ix for ix, row in ses1postqdf.iterrows()]

for ix, row in tempdf.iterrows():
    if ix in rowsin1:
        tempdf.drop(ix, inplace=True)
ses2postqdf = tempdf

ses1postqdf.reset_index(drop=True, inplace=True)
ses2postqdf.reset_index(drop=True, inplace=True)

In [26]:
# add prequestionnaire responses to corresponding subject
for ix, row in expdf.iterrows():
    for col in preqdf.columns:
        expdf.at[ix,col] = preqdf.at[ix,col]

# add session 1 postquestionnaire to first occurence of each subject
for ix, row in expdf.drop_duplicates('Subject ID').iterrows():
    expdf.loc[ix,'postqtime':] = (ses1postqdf.loc[ses1postqdf['Subject ID'] == row['Subject ID']]).drop(columns='Subject ID').values[0]

# add session 2 postquestionnaire to rest
for ix, row in expdf.iterrows():
    if np.isnan(row['postqtime']):
        expdf.loc[ix,'postqtime':] = (ses2postqdf.loc[ses2postqdf['Subject ID'] == row['Subject ID']]).drop(columns='Subject ID').values[0]

In [27]:
# drop subject -- self-reported difficuly with task due to sudden onset migraine and lack of sleep
expdf = expdf.loc[expdf['Subject ID'] != 'MD-102218-A-04'].reset_index(drop=True)

# drop subject -- reported technical issue with experiment audio
expdf = expdf.loc[expdf['Subject ID'] != 'MD-020719-B-01'].reset_index(drop=True)

# drop subjects who did not return for part 2
expdf = expdf.loc[expdf['Subject ID'] != 'MD-022019-B-04'].reset_index(drop=True)
expdf = expdf.loc[expdf['Subject ID'] != 'MD-022019-B-01'].reset_index(drop=True)

In [28]:
# drop subjects who didn't sufficiently complete task
expdf = expdf.loc[expdf['Subject ID'] != 'MD-101318-A-01'].reset_index(drop=True)
expdf = expdf.loc[expdf['Subject ID'] != 'MD-102218-B-06'].reset_index(drop=True)
expdf = expdf.loc[expdf['Subject ID'] != 'MD-013119-A-01'].reset_index(drop=True)
expdf = expdf.loc[expdf['Subject ID'] != 'MD-020119-B-03'].reset_index(drop=True)
expdf = expdf.loc[expdf['Subject ID'] != 'MD-020119-B-03'].reset_index(drop=True)

In [8]:
id_maps = pd.DataFrame(index=expdf['Subject ID'].unique(), columns=['session 1', 'session 2'])

for ix, row in expdf.iterrows():
    if pd.isnull(id_maps.loc[row['Subject ID'], 'session 1']):
        id_maps.loc[row['Subject ID'], 'session 1'] = row['uniqueid']
    else:
        id_maps.loc[row['Subject ID'], 'session 2'] = row['uniqueid']

In [3]:
# with open('../../../data/pickles/expdf.p', 'wb') as file:
#     pickle.dump(expdf, file)

In [17]:
# with open('../../../data/pickles/id_maps.p', 'wb') as file:
#     pickle.dump(id_maps, file)

In [213]:
with open('../../../data/pickles/expdf.p', 'rb') as file:
    expdf = pickle.load(file)

In [216]:
# # for checking correct mapping between experiment data and google forms
# for ix, row in expdf.iterrows():
#     for time in [row['datastring']['data'][0]['dateTime'], row['preqtime']]:
#         print(datetime.datetime.fromtimestamp(time/1e3))
#     print(ix)
#     print('___________')

2018-10-12 14:16:02.836000


TypeError: unsupported type for timedelta seconds component: numpy.int64

In [265]:
# for checking correct mapping between experiment data and google forms
x = []
y = []
for ix, row in expdf.iterrows():
    turktime, gformtime = row['datastring']['data'][0]['dateTime'], row['preqtime']
    ttime = datetime.datetime.fromtimestamp(turktime/1e3)
    x.append(turktime/1e3)
    gtime = datetime.datetime.fromtimestamp(gformtime/1e3)
    y.append(gformtime/1e3)
    print(ttime)
    print(gtime)
    print((ttime - gtime).total_seconds())
    print(ix)
    print('___________')
    if row['Subject ID'] == 'MD-102118-A-01':
        break

2018-10-12 14:16:02.836000
2018-10-12 14:15:43
19.836
0
___________
2018-10-12 15:19:16.776000
2018-10-12 15:18:53
23.776
1
___________
2018-10-12 15:29:26.510000
2018-10-12 15:28:58
28.51
2
___________
2018-10-12 16:23:41.845000
2018-10-12 16:23:22
19.845
3
___________
2018-10-12 16:31:57.256000
2018-10-12 16:31:26
31.256
4
___________
2018-10-12 17:29:45.466000
2018-10-12 17:29:08
37.466
5
___________
2018-10-12 17:32:42.852000
2018-10-12 17:32:24
18.852
6
___________
2018-10-13 14:35:15.708000
2018-10-13 14:35:01
14.708
7
___________
2018-10-13 15:37:13.664000
2018-10-13 15:36:58
15.664
8
___________
2018-10-13 15:40:05.736000
2018-10-13 15:39:29
36.736
9
___________
2018-10-13 16:31:30.137000
2018-10-13 16:31:02
28.137
10
___________
2018-10-13 16:38:57.045000
2018-10-13 16:38:42
15.045
11
___________
2018-10-13 17:36:34.814000
2018-10-13 17:36:09
25.814
12
___________
2018-10-13 17:47:13.097000
2018-10-13 17:46:55
18.097
13
___________
2018-10-13 18:38:30.153000
2018-10-13 18:38:1

In [266]:
y = []
for ix, row in expdf.iterrows():
    turktime, gformtime = row['datastring']['data'][0]['dateTime'], row['preqtime']
    ttime = datetime.datetime.fromtimestamp(turktime/1e3)
    x.append(turktime/1e3)
    gtime = datetime.datetime.fromtimestamp(gformtime/1e3)
    y.append(gformtime/1e3)
    print(ttime)
    print(gtime)
    print((ttime - gtime).total_seconds())
    print(ix)
    print('___________')
    if row['Subject ID'] == 'MD-102218-A-05':
        break

2018-10-12 14:16:02.836000
2018-10-12 14:15:43
19.836
0
___________
2018-10-12 15:19:16.776000
2018-10-12 15:18:53
23.776
1
___________
2018-10-12 15:29:26.510000
2018-10-12 15:28:58
28.51
2
___________
2018-10-12 16:23:41.845000
2018-10-12 16:23:22
19.845
3
___________
2018-10-12 16:31:57.256000
2018-10-12 16:31:26
31.256
4
___________
2018-10-12 17:29:45.466000
2018-10-12 17:29:08
37.466
5
___________
2018-10-12 17:32:42.852000
2018-10-12 17:32:24
18.852
6
___________
2018-10-13 14:35:15.708000
2018-10-13 14:35:01
14.708
7
___________
2018-10-13 15:37:13.664000
2018-10-13 15:36:58
15.664
8
___________
2018-10-13 15:40:05.736000
2018-10-13 15:39:29
36.736
9
___________
2018-10-13 16:31:30.137000
2018-10-13 16:31:02
28.137
10
___________
2018-10-13 16:38:57.045000
2018-10-13 16:38:42
15.045
11
___________
2018-10-13 17:36:34.814000
2018-10-13 17:36:09
25.814
12
___________
2018-10-13 17:47:13.097000
2018-10-13 17:46:55
18.097
13
___________
2018-10-13 18:38:30.153000
2018-10-13 18:38:1

In [278]:
expdf.loc[expdf['Subject ID'] == 'MD-102218-A-05'].loc[67]

uniqueid                                                                                                                                                              debugmmpS4:debugkthEs
datastring                                                                                                                                {'condition': 0, 'counterbalance': 0, 'assignm...
beginhit                                                                                                                                                         2018-10-28 16:36:02.235665
endhit                                                                                                                                                           2018-10-28 17:44:31.561434
hitid                                                                                                                                                                            debug8Vdvl
status                                                      

In [281]:
expdf.loc[expdf['Subject ID'] == 'MD-102118-A-01'].loc[66]

uniqueid                                                                                                                                                              debugoAeJs:debug47jDd
datastring                                                                                                                                {'condition': 0, 'counterbalance': 0, 'assignm...
beginhit                                                                                                                                                         2018-10-28 16:34:32.511568
endhit                                                                                                                                                           2018-10-28 17:52:26.238551
hitid                                                                                                                                                                            debugKK0U7
status                                                      

In [274]:
preqdf.columns

Index(['Timestamp', 'Subject ID',
       'Outside of this study, have you ever watched an episode of either of the TV shows "Atlanta" or "Arrested Development?"',
       'Is English your first language?',
       'Do you have any hearing or speech impairments?',
       'Do you have normal color vision?',
       'Are you taking any medications or have you had any recent injuries that could affect your memory or attention? (if so, describe below)',
       'If yes above, describe', 'In what year were you born?', 'Sex',
       'Ethnicity', 'Race (check all that apply)', 'Highest Degree Achieved',
       'If you are currently an undergraduate, what year are you?',
       'What is/was your major?',
       'How many hours of sleep did you get last night?',
       'How many cups of coffee have you had today?',
       'How alert are you feeling?'],
      dtype='object')

In [275]:
postqdf.columns

Index(['postqtime', 'Subject ID', 'How engaging did you find the episode?',
       'How easy/difficult was it to follow the episode?',
       'How well do you feel you recalled the events of the episode?',
       'How well do you feel you learned the characters' names over the course of the episode?',
       'How tired do you feel?'],
      dtype='object')